# Strands Agent Deployment in Bedrock AgentCore Runtime with Observability

This workshop demonstrates how to deploy Strands Agents with Amazon Bedrock AgentCore Runtime, integrating built-in tools and custom functions for comprehensive AI agent capabilities.

## Overview

In this lab, you will:
- Deploy the Strands Agents with tools to Bedrock AgentCore Runtime
- Use `boto3` with IAM permission to invoke the deployed agent
- Understand the nature of Bedrock AgentCore Runtime session
- Learn about GenAI Observability and Traceability

## Prerequisites

Before starting this lab, ensure you have:
- AWS credentials configured (IAM role or environment variables)
- Required Python packages installed
- Nova Pro model ID based on AWS region
- CloudWatch Transaction Search enabled for Bedrock AgentCore Observability

If you're not running in an environment with an IAM role assumed, set your AWS credentials as environment variables:

In [1]:
import os

#os.environ["AWS_ACCESS_KEY_ID"]=<YOUR ACCESS KEY>
#os.environ["AWS_SECRET_ACCESS_KEY"]=<YOUR SECRET KEY>
#os.environ["AWS_SESSION_TOKEN"]=<OPTIONAL - YOUR SESSION TOKEN IF TEMP CREDENTIAL>
#os.environ["AWS_REGION"]=<AWS REGION WITH BEDROCK AGENTCORE AVAILABLE>

Install required packages for Strands Agents and Bedrock AgentCore SDK:

In [2]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore bedrock-agentcore-starter-toolkit

Setup Nova Pro model ID based on AWS region:

In [3]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Nova Pro Model ID: us.amazon.nova-pro-v1:0


Enable [**CloudWatch APM → Transaction Search**](https://console.aws.amazon.com/cloudwatch/home#logsV2:transaction-search) for comprehensive observability and monitoring of your deployed Strands Agent. In this lab, set **X-Ray trace indexing to 100%** for generating all trace summaries for end-to-end transaction analytics.

![bedrock-agentcore-observability-setup](images/observability-setup.png)

## What is Strands Agent with Bedrock AgentCore Runtime?

Strands Agents provide a powerful framework for building AI agents with built-in and custom tool integrations. When deployed with Bedrock AgentCore Runtime, you get:

- **Scalable Deployment**: Managed infrastructure with auto-scaling capabilities
- **Secure Authentication**: Built-in security with Cognito integration
- **Observability**: CloudWatch integration for monitoring and debugging
- **Tool Integration**: Combine built-in tools like calculator with custom functions

In [4]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator

@tool
def weather(city: str) -> str:
    """Get weather information for a city
    Args:
        city: City or location name
    """
    return f"Weather for {city}: Sunny, 35°C" # dummy result for demo purpose

# Create and test the comprehensive Strands Agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt = """You are a helpful assistant that provides concise responses.
                    """,
    tools=[weather, calculator],
)

agent("How is the weather in HK? Return temperature in Fahrenheit")

<thinking> I need to get the weather information for Hong Kong and then convert the temperature from Celsius to Fahrenheit. First, I will use the weather tool to get the current temperature in Celsius. </thinking>

Tool #1: weather
<thinking> The current temperature in Hong Kong is 35°C. To convert this to Fahrenheit, I will use the formula F = C * 9/5 + 32. </thinking> 
Tool #2: calculator


╭────────────────────────────────────────────── Calculation Result ───────────────────────────────────────────────╮
│                                                                                                                 │
│  ╭───────────┬─────────────────────╮                                                                            │
│  │ Operation │ Evaluate Expression │                                                                            │
│  │ Input     │ 35 * 9/5 + 32       │                                                                            │
│  │ Result    │ 95                  │                                                                            │
│  ╰───────────┴─────────────────────╯                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The current temperature in Hong Kong is 95°F.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The current temperature in Hong Kong is 95°F.'}]}, metrics=EventLoopMetrics(cycle_count=3, tool_metrics={'weather': ToolMetrics(tool={'toolUseId': 'tooluse_xZwH28n1SVUo7ZUKQ1GQDW', 'name': 'weather', 'input': {'city': 'Hong Kong'}}, call_count=1, success_count=1, error_count=0, total_time=0.00045990943908691406), 'calculator': ToolMetrics(tool={'toolUseId': 'tooluse_WkHzcB50GX16BdxxLQ3nG9', 'name': 'calculator', 'input': {'mode': 'evaluate', 'expression': '35 * 9/5 + 32'}}, call_count=1, success_count=1, error_count=0, total_time=0.009983301162719727)}, cycle_durations=[0.41755223274230957], traces=[<strands.telemetry.metrics.Trace object at 0xffff69da97d0>, <strands.telemetry.metrics.Trace object at 0xffff69da2e10>, <strands.telemetry.metrics.Trace object at 0xffff69daa610>], accumulated_usage={'inputTokens': 5802, 'outputTokens': 143, 'totalTokens': 5945}, accumulated_metrics={'latencyMs': 2541}),

## Deploy Strands Agent to Bedrock AgentCore Runtime

Create a deployable version of our Strands Agent and deploy it as a managed service.
![bedrock-agentcore-runtime-launch](images/runtime-launch.png)

### Deployment Process:
1. Create Python file with agent configuration
2. Set up requirements.txt with dependencies
3. Configure AgentCore Runtime with authentication
4. Deploy using CodeBuild for containerization

### Step 1: Create Python File with Agent Configuration

Create the main Python file that defines our Strands Agent with built-in and custom tools. This file will be deployed to Bedrock AgentCore Runtime as a containerized service.

**Deployment Requirements:**
To deploy agents to AgentCore Runtime, we need to:
- Import the Runtime App: `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
- Initialize the App: `app = BedrockAgentCoreApp()`
- Decorate the invocation function with `@app.entrypoint`
- Let AgentCore Runtime control execution with `app.run()`

In [5]:
%%writefile strands_agent.py
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator
import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

# Setup Nova Pro model ID based on AWS region
NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
region = boto3.session.Session().region_name
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

@tool
def weather(city: str) -> str:
    """Get weather information for a city
    Args:
        city: City or location name
    """
    return f"Weather for {city}: Sunny, 35°C"


# Create and test the comprehensive Strands Agent
agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt = """You are a helpful assistant that provides concise responses.
                    """,
    tools=[weather, calculator],
)

@app.entrypoint
async def strands_agent_bedrock(payload, context):
    """
    Invoke the agent with a payload
    """
    print(f"Payload: {payload}")
    print(f"Context: {context}")
    user_input = payload.get("prompt", "No prompt found")
    response = agent(user_input)
    return response
    
    # Streaming Mode
    """
    stream = agent.stream_async(user_input)
    async for event in stream:
        if "data" in event:
            yield event
    """

if __name__ == "__main__":
    app.run()

Writing strands_agent.py


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

### Local Testing (Optional - skip if port 8080 is occupied in workshop environment)

Before deploying to AgentCore Runtime, you can test the agent locally to verify functionality.

**Start Local Server:**
```bash
cd 05-bedrock-agentcore-runtime-and-observability/
uv run strands_agent.py
```
or
```bash
cd 05-bedrock-agentcore-runtime-and-observability/
python strands_agent.py
```
The server will start at `http://localhost:8080`

**Test with cURL in another terminal:**
```bash
curl -X POST http://localhost:8080/invocations \
  -H "Content-Type: application/json" \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-User-Id: 123" \
  -H "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id: 1234567890123456789012345678901234567890" \
  -d '{"prompt": "How is the weather in HK?"}'
```

**Stop Local Server:**

Press `Ctrl+C` in terminal running `strands_agent.py` to stop the AgentCore Runtime running locally.

### Step 2: Create Requirements File

Define the Python dependencies needed for our Strands Agent deployment.

**Key Dependencies:**
- **aws-opentelemetry-distro**: Required for AgentCore observability
- **strands-agents**: Core Strands framework
- **bedrock-agentcore**: Runtime integration

**Observability Integration:**
The `aws-opentelemetry-distro` library enables automatic instrumentation for monitoring and tracing. As documented in the [AgentCore Observability Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html), the containerized environment (such as docker) need to add the following command:

```dockerfile
CMD ["opentelemetry-instrument", "python", "main.py"]
```

This auto-instrumentation approach automatically adds the OpenTelemetry SDK to the Python path for comprehensive observability.

In [6]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
bedrock-agentcore
bedrock-agentcore-starter-toolkit
boto3
aws-opentelemetry-distro>=0.10.0

Writing requirements.txt


### Step 3: Configure AgentCore Runtime

Set up the Bedrock AgentCore Runtime configuration with automatic resource creation.

**Generated Artifacts:**
This step creates essential deployment files:
- **Dockerfile**: Container configuration for the Strands Agent
- **.dockerignore**: list the excluded files when docker build
- **.bedrock_agentcore.yaml**: Runtime deployment configuration

Please note that when using the bedrock_agentcore_starter_toolkit to configure your agent, it takes care of the opentelemetry instrumentation. The generated Dockerfile will include:
```bash
CMD ["opentelemetry-instrument", "python", "runtime_agent_main.py"]
```

In [7]:
from bedrock_agentcore_starter_toolkit import Runtime
import boto3

region = boto3.session.Session().region_name
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_getting_started"
)
response

Entrypoint parsed: file=/workshop/05-bedrock-agentcore-runtime-and-observability/strands_agent.py, bedrock_agentcore_name=strands_agent
Configuring BedrockAgentCore agent: strands_getting_started


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

Generated .dockerignore
Generated Dockerfile: /workshop/05-bedrock-agentcore-runtime-and-observability/Dockerfile
Generated .dockerignore: /workshop/05-bedrock-agentcore-runtime-and-observability/.dockerignore
Setting 'strands_getting_started' as default agent
Bedrock AgentCore configured: /workshop/05-bedrock-agentcore-runtime-and-observability/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/workshop/05-bedrock-agentcore-runtime-and-observability/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/workshop/05-bedrock-agentcore-runtime-and-observability/Dockerfile'), dockerignore_path=PosixPath('/workshop/05-bedrock-agentcore-runtime-and-observability/.dockerignore'), runtime='None', region='us-west-2', account_id='315780440117', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### Step 4: Deploy to AgentCore Runtime

Launch the deployment process using AWS CodeBuild for containerization and deployment.

**Deployment Process:**
- Builds a containerized version of your Strands Agents
- Creates required AWS resources (ECR repository, IAM roles)
- Push container image to Amazon ECR
- Deploys to AgentCore Runtime as a managed, auto-scaling service

In [8]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_getting_started' to account 315780440117 (us-west-2)
Starting CodeBuild ARM64 deployment for agent 'strands_getting_started' to account 315780440117 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_getting_started
✅ ECR repository available: 315780440117.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-strands_getting_started
Getting or creating execution role for agent: strands_getting_started
Using AWS region: us-west-2, account ID: 315780440117
Role name: Amazo

Repository doesn't exist, creating new ECR repository: bedrock-agentcore-strands_getting_started


Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-235c86f826
Starting execution role creation process for agent: strands_getting_started
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-west-2-235c86f826
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-west-2-235c86f826
✓ Role created: arn:aws:iam::315780440117:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-235c86f826
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-strands_getting_started
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::315780440117:role/AmazonBedrockAgentCoreSDKRuntime-us-west-2-235c86f826
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: strands_getting_started
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-235c86f826
CodeBuild role doesn't exist, creating new role: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-235c86f826
Cre

### Step 5: Verify Deployment Status

Monitor the deployment progress and wait for the runtime to be ready:

In [9]:
import time

print("Checking AgentCore Runtime status...")
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
print(f"Initial status: {status}")

end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    print(f"Status: {status} - waiting...")
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']

if status == 'READY':
    print("✓ AgentCore Runtime is READY!")
else:
    print(f"⚠ AgentCore Runtime status: {status}")
    
print(f"Final status: {status}")

agent_runtime_id = launch_result.agent_id
agent_runtime_arn = launch_result.agent_arn
ecr_repo_name = launch_result.ecr_uri.split('/')[1]
codebuild_name = launch_result.codebuild_id.split(':')[0]
print(f"Strands AgentCore Runtime ID: {agent_runtime_id}")
print(f"Strands AgentCore Runtime ARN: {agent_runtime_arn}")
print(f"ECR Repo for Strands AgentCore Runtime: {ecr_repo_name}")
print(f"CodeBuild Project for Strands AgentCore Runtime: {codebuild_name}")

Retrieved Bedrock AgentCore status for: strands_getting_started


Checking AgentCore Runtime status...
Initial status: READY
✓ AgentCore Runtime is READY!
Final status: READY
Strands AgentCore Runtime ID: strands_getting_started-UA42Xt831g
Strands AgentCore Runtime ARN: arn:aws:bedrock-agentcore:us-west-2:315780440117:runtime/strands_getting_started-UA42Xt831g
ECR Repo for Strands AgentCore Runtime: bedrock-agentcore-strands_getting_started
CodeBuild Project for Strands AgentCore Runtime: bedrock-agentcore-strands_getting_started-builder


## Testing the Deployed Agent

Test the deployed Strands Agent by invoking it through the Bedrock AgentCore Runtime API with various prompts to verify all tools are working correctly.

In [10]:
import boto3
import json
import uuid

SESSION_ID = str(uuid.uuid4())

PROMPT = "How is the weather in HK? Return temperature in Fahrenheit"

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=boto3.session.Session().region_name
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID, #Provide same session identifier across multiple requests to maintain conversation context, and with better traceability
    payload=json.dumps({"prompt": PROMPT})
)

if "text/event-stream" in boto3_response.get("contentType", ""):
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line)
else:
    events = []
    for event in boto3_response.get("response", []):
        print(event.decode('utf-8'))
        events.append(event)

"The current temperature in Hong Kong is 95°F.\n"


### Testing Conversation History

Verify that the agent maintains conversation context across multiple requests using the same session ID:

In [11]:
PROMPT = "What did I ask?"

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_runtime_arn,
    qualifier="DEFAULT",
    runtimeSessionId=SESSION_ID, #Provide same session identifier across multiple requests to maintain conversation context, and with better traceability
    payload=json.dumps({"prompt": PROMPT})
)

if "text/event-stream" in boto3_response.get("contentType", ""):
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                print(line)
else:
    events = []
    for event in boto3_response.get("response", []):
        print(event.decode('utf-8'))
        events.append(event)

"You asked for the weather in Hong Kong and requested the temperature in Fahrenheit. The current temperature in Hong Kong is 95°F.\n"


## Conversation History and Session Management

**Strands Agents Conversation Management:**
Strands Agents includes built-in conversation management with a default `SlidingWindowConversationManager` strategy. This automatically maintains conversation context within session, but not persistent across multiple sessions.

**Reference:** [Strands Agents Conversation Management](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/agents/conversation-management/)

**AgentCore Runtime Session Isolation:**
- **Session Identification**: Is identified by a unique `runtimeSessionId` provided by your application, or by the Runtime itself in the first invocation if the `runtimeSessionId` is left empty
- **Session Isolation**: Runs in a dedicated microVM with completely isolated CPU, memory, and filesystem resources
- **Context perservation**: Preserves context across multiple interactions within the same conversation
- **Session Duration**: Session persist up to 8 hours maximum lifetime, or ended after 15 minutes due to inactivity
- **Automatic Cleanup**: After session termination, the entire microVM is terminated and memory is sanitized

**Reference:** [AgentCore Runtime Sessions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-sessions.html)


For persistent memory beyond session duration, integrate with **Bedrock AgentCore Memory** for long-term conversation storage and retrieval.

**Reference:** [AgentCore Memory: Add memory to your AI agent](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)

## AgentCore Observability on Amazon CloudWatch

### What is Bedrock AgentCore Observability?

Amazon Bedrock AgentCore provides built-in observability through CloudWatch and X-Ray integration. This enables monitoring of agent performance, tracing request flows, and analyzing conversation patterns.

Key features:
- **Session Tracking**: Monitor individual conversations
- **Distributed Tracing**: Track requests across components
- **Performance Metrics**: Latency, throughput, and error rates
- **Span Analysis**: Detailed execution breakdown

Learn more: [AgentCore Observability Guide](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)

### Viewing the Main Dashboard

Wait for 2-3 minutes, then access the [**Amazon CloudWatch console**](https://console.aws.amazon.com/cloudwatch/home#/gen-ai-observability/agent-core) to view your AgentCore observability dashboard:
![observability-main-dashboard.png](images/observability-main-dashboard.png)

### Session Management

Click **DEFAULT** in `strands_getting_started` to view session history. This shows our test session with two traces: "How is the weather in HK?" and "What did I ask?"
![observability-session-list.png](images/observability-session-list.png)

### Session Overview

Select a session to see metrics, trace timeline, and performance data:
![observability-trace-list.png](images/observability-trace-list.png)

### Trace Analysis

Click any trace to examine detailed execution steps:

**Span Details**:
![observability-trace-span-1.png](images/observability-trace-span-1.png)
![observability-trace-span-2.png](images/observability-trace-span-2.png)
![observability-trace-span-3.png](images/observability-trace-span-3.png)

## Resource Cleanup (Optional)

Clean up the deployed resources:

In [12]:
import boto3
import os

agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr',region_name=region)
codebuild_client = boto3.client('codebuild',region_name=region)

try:
    print("Deleting AgentCore Runtime...")
    agentcore_control_client.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
    print("✓ AgentCore Runtime deletion initiated")

    print("Deleting ECR repository...")
    ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
    print("✓ ECR repository deleted")

    print("Deleting CodeBuild Project...")
    codebuild_client.delete_project(name=codebuild_name)
    print("✓ CodeBuild Project deleted")

    print("Deleting Bedrock AgentCore configuration file...")
    os.remove(".bedrock_agentcore.yaml") 
    print("✓ .bedrock_agentcore.yaml deleted")
except Exception as e:
    print(f"❌ Error during cleanup: {e}")
    print("You may need to manually clean up some resources.")

Deleting AgentCore Runtime...
✓ AgentCore Runtime deletion initiated
Deleting ECR repository...
✓ ECR repository deleted
Deleting CodeBuild Project...
✓ CodeBuild Project deleted
Deleting Bedrock AgentCore configuration file...
✓ .bedrock_agentcore.yaml deleted


## Conclusion

In this lab, you successfully:

- ✅ Deployed a Strands Agent to Bedrock AgentCore Runtime
- ✅ Configured observability and monitoring with Amazon CloudWatch
- ✅ Tested end-to-end agent functionality with invoking remote agent in AgentCore Runtime environment
- ✅ Explored session isolation and automatic cleanup capabilities

## Key Benefits of AgentCore Runtime for Strands Agents

- **Production Deployment**: Scalable, managed infrastructure for AI agents
- **Comprehensive Observability**: Built-in monitoring, tracing, and debugging capabilities
- **Session Management**: Automatic session isolation and cleaned up 
- **Enterprise Ready**: Security, reliability, and compliance features for production workloads
